In [1]:
import faiss

In [2]:
import torch

In [3]:
import numpy as np

In [4]:
print("Torch version:", torch.__version__)
print("Built with CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch version: 2.11.0+cu128
Built with CUDA: 12.8
CUDA available: True
GPU count: 1
GPU: NVIDIA GeForce GTX 1650


In [5]:
from sentence_transformers import SentenceTransformer

D:\Anaconda3\envs\financial_analysis\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Reading metadata of the document chunks

In [6]:
import pickle

# Open the file in read-binary mode
with open('metadata.pkl', 'rb') as file:
    metadata = pickle.load(file)

In [7]:
chunk_lookup = {}

for chunk in metadata:
    key = (
        chunk["source"],
        chunk["chunk_index"]
    )
    chunk_lookup[key] = chunk
    
from collections import defaultdict
page_lookup = defaultdict(list)

for chunk in metadata:
    key = (
        chunk["source"],
        chunk["page"]
    )
    page_lookup[key].append(chunk)

# Loading saved embeddings of documents

In [9]:
index = faiss.read_index("docs.index")

# Loading Encoder Models

In [10]:
model = SentenceTransformer(
    "BAAI/bge-base-en-v1.5",
    device="cuda"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2727.01it/s]


In [11]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "BAAI/bge-reranker-base",
    device="cuda"
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2846.68it/s]


# Retrieval Functions

In [12]:
import numpy as np
from collections import Counter
from sentence_transformers import CrossEncoder


# ---------------------------------------------------------
# 1. Encode query
# ---------------------------------------------------------

def encode_query(query, embedding_model):
    """
    Encode a query using the embedding model.

    Parameters
    ----------
    query : str
        User query.
    embedding_model : SentenceTransformer
        Loaded embedding model.

    Returns
    -------
    np.ndarray
        Normalized query embedding.
    """
    return embedding_model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    )


# ---------------------------------------------------------
# 2. Retrieve candidates from FAISS
# ---------------------------------------------------------

def retrieve_candidates(query_vector, index, metadata, k=50):
    """
    Retrieve top-k candidate chunks from FAISS.

    Parameters
    ----------
    query_vector : np.ndarray
        Encoded query vector.
    index : faiss.Index
        FAISS index.
    metadata : list
        Metadata corresponding to FAISS vectors.
    k : int, default=50
        Number of candidates to retrieve.

    Returns
    -------
    distances : np.ndarray
        FAISS similarity/distance scores.
    candidate_chunks : list
        Retrieved metadata chunks.
    """
    distances, indices = index.search(query_vector, k)

    candidate_chunks = [
        metadata[i]
        for i in indices[0]
    ]

    return distances[0], candidate_chunks


# ---------------------------------------------------------
# 3. Rerank candidates using CrossEncoder
# ---------------------------------------------------------

def rerank_candidates(
    query,
    candidate_chunks,
    reranker
):
    """
    Rerank retrieved chunks using a CrossEncoder.

    Parameters
    ----------
    query : str
        User query.
    candidate_chunks : list
        Retrieved chunks.
    reranker : CrossEncoder
        Loaded CrossEncoder reranker.

    Returns
    -------
    np.ndarray
        Reranker scores.
    """
    pairs = [
        (query, chunk["text"])
        for chunk in candidate_chunks
    ]

    scores = reranker.predict(pairs)

    return np.asarray(scores)


# ---------------------------------------------------------
# 4. Calculate metadata scores
# ---------------------------------------------------------

def calculate_metadata_scores(query, candidate_chunks):
    """
    Calculate metadata-based scores.

    Logic preserved from the original implementation:

    - Same-document boost
    - Heading word-overlap boost

    Parameters
    ----------
    query : str
        User query.
    candidate_chunks : list
        Retrieved chunks.

    Returns
    -------
    np.ndarray
        Normalized metadata scores.
    """

    # Count how many retrieved chunks belong
    # to each document.
    doc_counts = Counter(
        chunk["source"]
        for chunk in candidate_chunks
    )

    max_count = max(doc_counts.values())

    query_words = set(
        query.lower().split()
    )

    metadata_scores = []

    for chunk in candidate_chunks:

        score = 0.0

        # ---------------------------------------------
        # Same-document boost
        # ---------------------------------------------
        score += (
            doc_counts[chunk["source"]]
            / max_count
        )

        # ---------------------------------------------
        # Heading boost
        # ---------------------------------------------
        if "heading" in chunk:

            heading_words = set(
                chunk["heading"].lower().split()
            )

            overlap = len(
                query_words & heading_words
            )

            score += (
                overlap /
                max(1, len(query_words))
            )

        metadata_scores.append(score)

    metadata_scores = np.asarray(
        metadata_scores
    )

    # ---------------------------------------------
    # Normalize metadata scores
    # ---------------------------------------------
    metadata_scores = normalize_scores(
        metadata_scores
    )

    return metadata_scores


# ---------------------------------------------------------
# 5. Normalize scores
# ---------------------------------------------------------

def normalize_scores(scores):
    """
    Min-max normalize scores to [0, 1].

    Same normalization logic used for
    FAISS and reranker scores.
    """

    scores = np.asarray(scores)

    return (
        scores - scores.min()
    ) / (
        scores.max() -
        scores.min() +
        1e-8
    )


# ---------------------------------------------------------
# 6. Fuse FAISS + reranker + metadata scores
# ---------------------------------------------------------

def fuse_and_rank(
    faiss_scores,
    rerank_scores,
    metadata_scores,
    candidate_chunks,
    faiss_weight=0.60,
    rerank_weight=0.25,
    metadata_weight=0.15,
    top_k=10
):
    """
    Fuse FAISS, reranker, and metadata scores.

    Default weights preserve the original logic:

        0.60 * FAISS
        0.25 * reranker
        0.15 * metadata

    Parameters
    ----------
    faiss_scores : np.ndarray
        FAISS scores.
    rerank_scores : np.ndarray
        CrossEncoder scores.
    metadata_scores : np.ndarray
        Metadata scores.
    candidate_chunks : list
        Retrieved chunks.
    top_k : int
        Number of final results.

    Returns
    -------
    list
        Top-ranked chunks.
    """

    # Normalize FAISS scores
    faiss_scores = normalize_scores(
        faiss_scores
    )

    # Normalize reranker scores
    rerank_scores = normalize_scores(
        rerank_scores
    )

    # ---------------------------------------------
    # Weighted fusion
    # ---------------------------------------------
    final_scores = (
        faiss_weight * faiss_scores
        + rerank_weight * rerank_scores
        + metadata_weight * metadata_scores
    )

    # ---------------------------------------------
    # Sort by final score
    # ---------------------------------------------
    ranked = sorted(
        zip(final_scores, candidate_chunks),
        key=lambda x: x[0],
        reverse=True
    )

    return ranked[:top_k]


# ---------------------------------------------------------
# 7. Expand results using page lookup
# ---------------------------------------------------------

def expand_results(results, page_lookup):
    """
    Expand retrieved chunks using page_lookup.

    Logic preserved from the original implementation:
    each result is expanded using (source, page).

    Parameters
    ----------
    results : list
        Ranked results.
    page_lookup : dict
        Dictionary keyed by (source, page).

    Returns
    -------
    list
        Expanded page-level results.
    """

    expanded_results = []

    for _, chunk in results:

        key = (
            chunk["source"],
            chunk["page"]
        )

        expanded_results.extend(
            page_lookup[key]
        )

    return expanded_results


# ---------------------------------------------------------
# 8. Build final context
# ---------------------------------------------------------

def build_context(expanded_results):
    """
    Convert expanded results into a single
    context string for the LLM.
    """

    context_parts = []

    for result in expanded_results:

        context_parts.append(
            f"""
Source: {result['source']}
Page: {result['page']}

{result['text']}

---
"""
        )

    return "\n".join(context_parts)

In [41]:
def retrieve_context(
    query,
    model,
    index,
    metadata,
    reranker,
    page_lookup,
    retrieval_k=10,
    final_k=10
):
    """
    Complete RAG retrieval pipeline.

    Pipeline:

        Query
          ↓
        Embedding
          ↓
        FAISS retrieval
          ↓
        CrossEncoder reranking
          ↓
        Metadata scoring
          ↓
        Weighted fusion
          ↓
        Top-K
          ↓
        Page expansion
          ↓
        Context
    """

    # 1. Encode query
    query_vector = encode_query(
        query,
        model
    )

    # 2. FAISS retrieval
    faiss_scores, candidate_chunks = (
        retrieve_candidates(
            query_vector,
            index,
            metadata,
            k=retrieval_k
        )
    )

    # 3. CrossEncoder reranking
    rerank_scores = rerank_candidates(
        query,
        candidate_chunks,
        reranker
    )

    # 4. Metadata scoring
    metadata_scores = calculate_metadata_scores(
        query,
        candidate_chunks
    )

    # 5. Score fusion + ranking
    ranked_results = fuse_and_rank(
        faiss_scores=faiss_scores,
        rerank_scores=rerank_scores,
        metadata_scores=metadata_scores,
        candidate_chunks=candidate_chunks,
        faiss_weight=0.60,
        rerank_weight=0.25,
        metadata_weight=0.15,
        top_k=final_k
    )

    # 6. Expand retrieved pages
    expanded_results = expand_results(
        ranked_results,
        page_lookup
    )

    # 7. Build LLM context
    context = build_context(
        expanded_results
    )

    return {
        "query": query,
        "candidate_chunks": candidate_chunks,
        "ranked_results": ranked_results,
        "expanded_results": expanded_results,
        "context": context
    }

In [28]:
[res['chunk_index'] for res in results["candidate_chunks"]]

[95, 135, 127, 132, 136, 67, 121, 184, 72, 122]

In [27]:
query = "what are some key exercises to strengthen back muscles?"

results = retrieve_context(
    query=query,
    model=model,
    index=index,
    metadata=metadata,
    reranker=reranker,
    page_lookup=page_lookup,
    retrieval_k=10,
    final_k=10
)

context = results["context"]

print(context)


Source: D:\RAG\pdf files\Matt Furey - Combat Conditioning (1)_text.pdf
Page: 33

Wall Walking 
This exercise Is another one thal siretches and strengthens all the muscles along the 
spine. E also works the abdorninals as they involuntarily contract when you bend backwards. 
increased Hexlhilty and strength in the spine goes a long way toward Increasing energy levels 
and Improving overall health, 
1. Stand with your back and heels Hat against the wall, 
2. Take two steps, heel to tos, until you are three feel fram the wall

---


Source: D:\RAG\pdf files\Matt Furey - Combat Conditioning (1)_text.pdf
Page: 33

2. Take two steps, heel to tos, until you are three feel fram the wall 
3. From there, lean backward with your hands stretched above your head, 
4, Slowly move your hands down the wall, Continue walking until the top of your 
head lightly touches the flow. 
5. Turn to your stomach and stand up again. 
& Do five to ten repetitions. 
z 
2 
Breathe naturally.

---


Source: D:\RAG\p

# Generation

In [15]:
from ollama import chat

In [22]:
def generate_llm_response(query, context, model="llama3.2:3b"):
    prompt = f"""
            You are a helpful assistant answering questions from documents.

            Rules:
            - Answer only from the provided context.
            - If the answer is not present, say "I could not find that information."
            - Quote important facts when possible.
            - Mention the source document if available.

            Context:
            {context}

            Question:
            {query}

            Answer:
            """
    
    response = chat(
        model=model,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )
    
    return response["message"]["content"].strip()


In [23]:
%%time
context = results["context"]
response = generate_llm_response(query, context)
# print(response["message"]["content"])
print(response)

Here are some key exercises mentioned in the text:

1. **Back Bridge**: The greatest exercise in Combat Conditioning, which strengthens muscles in the abdominals, legs, hips, buttocks, back, shoulders, and neck.
2. **Kneeling Back Bend**: Develops flexibility and strength throughout the back and thighs, with a focus on hip flexors and buttocks.
3. **Leg Lifts Behind Head**: Strengthens abdominals, lower back, and hip flexors, while also stretching the spine, shoulders, and upper back.
4. **Reverse Leg Lifts**: Develops strength in the abdominals, lower back, and buttocks, with a good stretch for the lower back.

These exercises are designed to improve flexibility, strength, and overall muscle development in the back muscles.
CPU times: total: 31.2 ms
Wall time: 23.8 s


# Evaluation

In [19]:
import json
import math
import time
import pandas as pd

In [20]:
EVAL_FILE = "rag_evaluation_dataset.json"

OLLAMA_MODEL = "llama3.2:3b"

K = 10

OUTPUT_FILE = "rag_eval_results.csv"


# ============================================================
# LOAD EVALUATION DATASET
# ============================================================

with open(
    EVAL_FILE,
    "r",
    encoding="utf-8"
) as f:

    evaluation_dataset = json.load(f)


print(
    f"Loaded {len(evaluation_dataset)} questions"
)

Loaded 38 questions


In [30]:
# ============================================================
# RECALL@10
# ============================================================

def recall_at_k(
    retrieved_ids,
    gold_ids,
    k=10
):

    retrieved = set(
        retrieved_ids[:k]
    )

    gold = set(
        gold_ids
    )

    if not gold:
        return 0.0

    return len(
        retrieved & gold
    ) / len(gold)


# ============================================================
# nDCG@10
# ============================================================

def ndcg_at_k(
    retrieved_ids,
    gold_ids,
    k=10
):

    gold = set(
        gold_ids
    )

    if not gold:
        return 0.0

    # --------------------------------------------------------
    # DCG
    # --------------------------------------------------------

    dcg = 0.0

    for rank, chunk_id in enumerate(
        retrieved_ids[:k],
        start=1
    ):

        relevance = (
            1
            if chunk_id in gold
            else 0
        )

        dcg += (
            (2 ** relevance - 1)
            /
            math.log2(rank + 1)
        )

    # --------------------------------------------------------
    # Ideal DCG
    # --------------------------------------------------------

    ideal_relevant = min(
        len(gold),
        k
    )

    idcg = sum(
        (2 ** 1 - 1)
        /
        math.log2(rank + 1)

        for rank in range(
            1,
            ideal_relevant + 1
        )
    )

    if idcg == 0:
        return 0.0

    return dcg / idcg


# ============================================================
# LLM ANSWER QUALITY
# ============================================================

def evaluate_answer(
    question,
    ground_truth,
    generated_answer
):

    prompt = f"""
You are evaluating the quality of a RAG answer.

QUESTION:
{question}

REFERENCE ANSWER:
{ground_truth}

GENERATED ANSWER:
{generated_answer}

Give ONE overall score from 1 to 5.

5 = Excellent
    Correct, complete and well supported.

4 = Good
    Mostly correct with only minor omissions.

3 = Acceptable
    Partially correct but has noticeable omissions.

2 = Poor
    Major errors or missing information.

1 = Very poor
    Incorrect or does not answer the question.

Return ONLY JSON:

{{
    "score": 1,
    "reason": "short explanation"
}}
"""

    response = chat(
        model=OLLAMA_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        format="json"
    )

    return json.loads(
        response["message"]["content"]
    )



In [ ]:
# ============================================================
# EVALUATE
# ============================================================

results = []


for i, item in enumerate(evaluation_dataset, start=1):

    print("\n" + "=" * 70)
    query = item["question"]

    print(f"Question {i}/{len(evaluation_dataset)}")
    print(query)
    print(f"Ground truth: {item["ground_truth_answer"]}")

    # --------------------------------------------------------
    # RETRIEVAL
    # --------------------------------------------------------

    start = time.perf_counter()
    

    retrieved = retrieve_context(
        query=query,
        model=model,
        index=index,
        metadata=metadata,
        reranker=reranker,
        page_lookup=page_lookup,
        retrieval_k=K,
        final_k=K
    )
    retrieval_time = (time.perf_counter() - start)
    retrieved_ids = [res['chunk_index'] for res in retrieved["candidate_chunks"]]

    # --------------------------------------------------------
    # RETRIEVAL METRICS
    # --------------------------------------------------------

    recall = recall_at_k(
        retrieved_ids,
        item["gold_chunks"],
        K
    )

    ndcg = ndcg_at_k(
        retrieved_ids,
        item["gold_chunks"],
        K
    )

    # --------------------------------------------------------
    # GENERATE ANSWER
    # --------------------------------------------------------

    context = retrieved["context"]
    answer = generate_llm_response(query, context)

    # --------------------------------------------------------
    # ANSWER QUALITY
    # --------------------------------------------------------

    evaluation = evaluate_answer(
        item["question"],
        item["ground_truth_answer"],
        answer
    )
    answer_quality = evaluation["score"]
    print(f"Generated answer: {answer}")

    # --------------------------------------------------------
    # STORE
    # --------------------------------------------------------

    result = {
        "question_id": item["id"],
        "question": item["question"],
        "type": item.get("type"),
        "source": item["source"],
        "recall@10": recall,
        "ndcg@10": ndcg,
        "answer_quality": answer_quality,
        "retrieval_time": retrieval_time,
        "generated_answer": answer,
        "ground_truth_answer": item["ground_truth_answer"],
        "judge_reason": evaluation["reason"]
    }
    results.append(result)

    # --------------------------------------------------------
    # PRINT
    # --------------------------------------------------------

    print(
        f"Recall@10       : {recall:.3f}"
    )

    print(
        f"nDCG@10         : {ndcg:.3f}"
    )

    print(
        f"Answer Quality  : "
        f"{answer_quality}/5"
    )



In [45]:
# ============================================================
# SUMMARY
# ============================================================

df = pd.DataFrame(
    results
)


print("\n")
print("=" * 70)
print("RAG EVALUATION SUMMARY")
print("=" * 70)

print(
    f"Recall@10       : "
    f"{df['recall@10'].mean():.3f}"
)

print(
    f"nDCG@10         : "
    f"{df['ndcg@10'].mean():.3f}"
)

print(
    f"Answer Quality  : "
    f"{df['answer_quality'].mean():.2f}/5"
)






RAG EVALUATION SUMMARY
Recall@10       : 0.518
nDCG@10         : 0.492
Answer Quality  : 3.68/5


In [46]:
# ============================================================
# SAVE
# ============================================================

df.to_csv(
    OUTPUT_FILE,
    index=False
)

print(
    f"\nSaved results to: {OUTPUT_FILE}"
)


Saved results to: rag_eval_results.csv
